In [1]:
%pip install --upgrade pip 
%pip install --upgrade transformers datasets[audio] accelerate

Note: you may need to restart the kernel to use updated packages.
zsh:1: no matches found: datasets[audio]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
# path to ffmpeg bin directory(C:\LLM_Project\ffmpeg\bin)
# os.environ["PATH"] += os.pathsep + r"C:\LLM_Project\ffmpeg\bin"  
# 맥북(macOS)에서 Homebrew로 설치한 FFmpeg 경로 추가
# (Homebrew 기본 경로: /opt/homebrew/bin 또는 /usr/local/bin)
ffmpeg_bin_path = "/opt/homebrew/bin" # Apple Silicon(M1/M2/M3 등) 맥북 기준
if ffmpeg_bin_path not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + ffmpeg_bin_path

In [17]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio transformers accelerate "datasets[audio]"

In [ ]:
import torch

from transformers import AutoModelForSpeechSeq2Seq
from transformers import AutoProcessor
from transformers import pipeline
#from datasets import load_dataset  # Using Audio_Boohwal.mp3

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"  # you can change to other Whisper models

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(        #Create a pipeline for speech recognition
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True, # to get word-level timestamps
    chunk_length_s=10,      # long audio files -> chunk size(10 seconds) as needed
    stride_length_s=2,      # overlapping chunks(2 seconds) to avoid missing words   
)

#dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation") 
#sample = dataset[0]["audio"]        # example from dataset
#sample = "./Data/Audio_Boohwal.mp3"  # local audio file path
sample = "../Data/STT_Audio_58s.mp3"  # local audio file path

result = pipe(sample)
#print(result["text"])

print(result)   # print full result with word-level timestamps

In [23]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = "cpu"
torch_dtype = torch.float32

# 용량이 작은 모델로 먼저 테스트 (다운로드 및 실행 확인용)
model_id = "openai/whisper-small"

print("1. 가벼운 Whisper 모델 로딩 중...")
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
print("모델 로드 성공!")

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,
    chunk_length_s=10,
    stride_length_s=2,
)

sample = "/Users/yang-yeeun/Desktop/생성형AI프로그래밍/LLM_Project/Data/STT_Audio_58s.mp3"

print("2. STT 실행 중...")
result = pipe(sample)
print("--- 완료 ---")
print(result)

1. 가벼운 Whisper 모델 로딩 중...


Loading weights: 100%|██████████| 479/479 [00:00<00:00, 8181.49it/s]


모델 로드 성공!


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


2. STT 실행 중...


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece token

--- 완료 ---
{'text': ' 안녕하세요. 이 강의는 GPT API로 체법 만들기라는 내용을 다루는 강입니다. GPT API에 대해서 생소하신 분들도 있을텐데, 우리가 잘 알고 있는 채치 GPT, 채치 GPT를 이용해, 채치 GPT의 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 뭐 이런 강의들이 사실 많이 있습니다. 그래서 뭐 여러가지들이 있는데 좀 이 강의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 제법 프로그램을 만드는 게 사실 쉽지는 않은는데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로는 여러가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 그래서 프로그램이 실행되는 모습을 한번 보여 드릴게요 우리가 만들 프로그램은 이런식으로 이제 나타나게 되고', 'chunks': [{'timestamp': (0.0, 6.88), 'text': ' 안녕하세요. 이 강의는 GPT API로 체법 만들기라는 내용을 다루는 강입니다.'}, {'timestamp': (6.88, 27.56), 'text': ' GPT API에 대해서 생소하신 분들도 있을텐데, 우리가 잘 알고 있는 채치 GPT, 채치 GPT를 이용해, 채치 GPT의 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 뭐 이런 강의들이 사실 많이 있습니다. 그래서 뭐 여러가지들이 있는데 좀 이 강의 특징이라고 한다면'}, {'timestamp': (27.56, 35.22), 'text': ' GPT로 명확한 미션을 달성하는 제법 프로그램을 만드는 게 사실 쉽지는 않은는데 이걸 어떻게 해서 구현을 하는지'}, {'timestamp': (35.22, 49.8), 'text': ' 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로는 여러가지가 될 수

In [24]:
# Save Chunks to CSV file
start_end_text = [] # extract start, end, text from each chunk

for chunk in result["chunks"]:
    start = chunk["timestamp"][0]
    end = chunk["timestamp"][1]
    text = chunk["text"]
    start_end_text.append([start, end, text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
df.to_csv("STT_Audio_Chunks.csv", index=False, sep="|")
display(df)

,start,end,text
0,0.00,6.88,안녕하세요. 이 강의는 GPT API로 체법 만들기라는 내용을 다루는 강입니다.
1,6.88,27.56,"GPT API에 대해서 생소하신 분들도 있을텐데, 우리가 잘 알고 있는 채치 GP..."
2,27.56,35.22,GPT로 명확한 미션을 달성하는 제법 프로그램을 만드는 게 사실 쉽지는 않은는데 ...
3,35.22,49.80,그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로는 여러가지가...
4,49.80,54.34,그래서 프로그램이 실행되는 모습을 한번 보여 드릴게요 우리가 만들
5,54.34,58.16,프로그램은 이런식으로 이제 나타나게 되고
